In [ ]:
import pandas as pd
from rapidfuzz import fuzz
import re

# ============================================================
# 1. Load Excel File
# ============================================================
df = pd.read_excel(
    r"D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\Main Mapping\Name Mapping Excel.xlsx",
    sheet_name="Sheet1"
)

# ============================================================
# 2. Stopwords, essential tokens & rules
# ============================================================
STOPWORDS = {"THE", "LTD", "PVT", "CO", "AND", "LLP"}
ESSENTIALS = {"ENERGY", "ELECTRIC"}
REGIONAL_WORDS = {"NORTH", "SOUTH", "EAST", "WEST", "UTTAR", "DAKSHIN"}

INSURANCE_LIFE = {"LIFE"}
INSURANCE_GENERAL = {"GENERAL", "GI", "INSURANCE"}

TELECOM_WORDS = {"AIRTEL", "VODAFONE", "IDEA", "JIO", "BSNL", "AIRCEL"}
BANK_WORDS = {"BANK", "FINANCE", "NBFC"}
MESSAGE_WORDS = {"MESSAGE", "MSG", "SMS"}
SERVICE_WORDS = {"OPERATIONS", "SUPPORT", "SERVICE", "SERVICES", "SOLUTIONS"}

TITAN_BRAND = {"TITAN"}

REPLACEMENTS = {
    "PVT.": "PVT",
    "PRV": "PVT",
    "PRIVATE": "PVT",
    "LIMITED": "LTD",
    "&": "AND",
    "CO.": "CO",
    "TECHNOLOGY": "TECH",
    "TECHNOLOGIES": "TECH",
}

# ============================================================
# 3. Cleaning Functions
# ============================================================
def normalize_variants(s):
    for k, v in REPLACEMENTS.items():
        s = re.sub(rf"\b{k}\b", v, s)
    return s

def clean_text(s):
    if pd.isna(s):
        return None
    s = str(s).upper().strip()
    s = normalize_variants(s)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

def strip_stopwords(s):
    if not s:
        return None
    tokens = s.split()
    tokens = [t for t in tokens if t not in STOPWORDS or t in ESSENTIALS]
    return " ".join(tokens) if tokens else s

def has_conflicting_region(a, b):
    a_tokens = set(a.split())
    b_tokens = set(b.split())
    for w in REGIONAL_WORDS:
        if (w in a_tokens) != (w in b_tokens):
            return True
    return False

def insurance_conflict(a, b):
    a_tokens = set(a.split())
    b_tokens = set(b.split())
    if (INSURANCE_LIFE & a_tokens and INSURANCE_GENERAL & b_tokens) or \
       (INSURANCE_LIFE & b_tokens and INSURANCE_GENERAL & a_tokens):
        return True
    return False

# ============================================================
# 4. TOKEN-BASED SCORING ENGINE
# ============================================================
def token_based_score(a, b):
    a_tokens = a.split()
    b_tokens = b.split()
    token_matches = []
    for t1 in a_tokens:
        best = 0
        for t2 in b_tokens:
            best = max(best, fuzz.ratio(t1, t2))
        token_matches.append(best)
    weights = [1.5] + [1.0] * (len(token_matches) - 1)
    weighted_sum = sum(t * w for t, w in zip(token_matches, weights))
    max_possible = sum(weights) * 100
    return (weighted_sum / max_possible) * 100

# ============================================================
# 5. FINAL SCORING ENGINE (HARD RULES + TOKEN ENGINE)
# ============================================================
def final_match_score(a, b):
    if not a or not b:
        return 0

    a_tokens = a.split()
    b_tokens = b.split()
    a_set = set(a_tokens)
    b_set = set(b_tokens)

    a1 = a_tokens[0]
    b1 = b_tokens[0]
    a2 = a_tokens[1] if len(a_tokens) > 1 else ""
    b2 = b_tokens[1] if len(b_tokens) > 1 else ""

    # HARD RULES
    if has_conflicting_region(a, b):
        return 0
    if insurance_conflict(a, b):
        return 0
    if "BANK" in a_set and "BANK" in b_set:
        if fuzz.ratio(a2, b2) < 90:
            return 0
    if ("BANK" in a_set and SERVICE_WORDS & b_set) or \
       ("BANK" in b_set and SERVICE_WORDS & a_set):
        return 0
    if a1 in MESSAGE_WORDS and b1 in MESSAGE_WORDS:
        if fuzz.ratio(a2, b2) < 90:
            return 0
    if a1 in TELECOM_WORDS and b1 in TELECOM_WORDS:
        if fuzz.ratio(a1, b1) < 95:
            return 0
    if a1 == "TITAN" and b1 == "TITAN":
        if fuzz.ratio(a2, b2) < 90:
            return 0
    if fuzz.ratio(a1, b1) < 85:
        return 0
    if a2 and b2 and fuzz.ratio(a2, b2) < 80:
        return 0

    # TOKEN + FUZZY SCORE
    token_score = token_based_score(a, b)
    fuzzy_score = (
        0.5 * fuzz.token_set_ratio(a, b) +
        0.3 * fuzz.partial_ratio(a, b) +
        0.2 * fuzz.token_sort_ratio(a, b)
    )
    final_score = 0.7 * token_score + 0.3 * fuzz.token_set_ratio(a, b)

    # Substring booster
    if a in b or b in a:
        final_score += 10

    return min(100, final_score)

# ============================================================
# 6. MATCHING FUNCTION (RETURN ORIGINAL VALUES)
# ============================================================
def find_best_match(main_name, cleaned_choices, original_choices, threshold=80):
    if not main_name:
        return None, 0, "No Match"

    main_clean = strip_stopwords(clean_text(main_name))
    best_score = 0
    best_index = None

    for i, clean_choice in enumerate(cleaned_choices):
        score = final_match_score(main_clean, clean_choice)
        if score > best_score:
            best_score = score
            best_index = i

    if best_score >= threshold:
        return original_choices[best_index], best_score, "High Confidence"
    return None, best_score, "No Match"

# ============================================================
# 7. Prepare choice sets (CLEANED FOR INTERNAL MATCHING ONLY)
# ============================================================
# Original values (for output)
original_radhika = df["Parent company Sheet 2 Radhika"].dropna().tolist()
original_automation = df["Automation File"].dropna().tolist()
original_sms = df["SMS Names2"].dropna().tolist()
original_zoho = df["Zoho"].dropna().tolist()

# Cleaned copies for internal matching only
clean_radhika = [strip_stopwords(clean_text(x)) for x in original_radhika]
clean_automation = [strip_stopwords(clean_text(x)) for x in original_automation]
clean_sms = [strip_stopwords(clean_text(x)) for x in original_sms]
clean_zoho = [strip_stopwords(clean_text(x)) for x in original_zoho]

choices_dict_clean = {
    "Radhika": clean_radhika,
    "Automation": clean_automation,
    "SMS": clean_sms,
    "Zoho": clean_zoho,
}

choices_dict_original = {
    "Radhika": original_radhika,
    "Automation": original_automation,
    "SMS": original_sms,
    "Zoho": original_zoho,
}

thresholds = {"Radhika": 80, "Automation": 80, "SMS": 80, "Zoho": 80}


results = []

for name in df["Main Names"]:
    row = {"Main Name": name}  # Original main name

    for col in choices_dict_clean:
        match, score, note = find_best_match(
            name,
            choices_dict_clean[col],
            choices_dict_original[col],
            thresholds[col]
        )
        row[f"{col} Match"] = match           
        row[f"{col} Score"] = score
        row[f"{col} Note"] = note

    results.append(row)

results_df = pd.DataFrame(results)

output_path = r"D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\python result\matched_results_test1.xlsx"
results_df.to_excel(output_path, index=False)

print("✅ Matching complete.")
print(f"Saved to: {output_path}")


✅ Matching complete.
Saved to: D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\python result\matched_results_test1.xlsx


In [5]:
import pandas as pd
from rapidfuzz import fuzz
import re

# ============================================================
# 1. Load Data
# ============================================================
df = pd.read_excel(
    r"D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\Main Mapping\Name Mapping Excel.xlsx",
    sheet_name="Sheet1"
)

# ============================================================
# 2. Basic Normalization Rules
# ============================================================
STOPWORDS = {"THE", "LTD", "PVT", "CO", "AND", "LLP"}
REPLACEMENTS = {
    "PVT.": "PVT",
    "PRIVATE": "PVT",
    "LIMITED": "LTD",
    "&": "AND",
    "CO.": "CO",
    "TECHNOLOGY": "TECH",
    "TECHNOLOGIES": "TECH"
}

def normalize_variants(s):
    for k, v in REPLACEMENTS.items():
        s = re.sub(rf"\b{k}\b", v, s)
    return s

def clean_text(s):
    if pd.isna(s):
        return None
    s = str(s).upper().strip()
    s = normalize_variants(s)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else None

def strip_stopwords(s):
    if not s:
        return None
    tokens = s.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return " ".join(tokens) if tokens else s

# ============================================================
# 3. Simple Scoring Function
# ============================================================
def match_score(a, b):
    """Simplified scoring: token_set + token_sort + partial fuzzy."""
    if not a or not b:
        return 0
    a = strip_stopwords(clean_text(a))
    b = strip_stopwords(clean_text(b))
    
    s1 = fuzz.token_set_ratio(a, b)
    s2 = fuzz.token_sort_ratio(a, b)
    s3 = fuzz.partial_ratio(a, b)
    
    return (0.5*s1 + 0.3*s2 + 0.2*s3)

# ============================================================
# 4. Find Best Match (Simple)
# ============================================================
def find_best_match(main_name, cleaned_choices, original_choices, threshold=80):
    if not main_name:
        return None, 0, "No Match"

    main_clean = strip_stopwords(clean_text(main_name))
    best_score = 0
    best_index = None

    for i, choice_clean in enumerate(cleaned_choices):
        score = match_score(main_clean, choice_clean)
        if score > best_score:
            best_score = score
            best_index = i

    if best_score >= threshold:
        return original_choices[best_index], best_score, "High Confidence"
    return None, best_score, "No Match"

# ============================================================
# 5. Prepare Clean Lists
# ============================================================
def clean_list(col):
    original = df[col].dropna().tolist()
    cleaned = [strip_stopwords(clean_text(x)) for x in original]
    return original, cleaned

original_radhika, clean_radhika = clean_list("Parent company Sheet 2 Radhika")
original_aut, clean_aut = clean_list("Automation File")
original_sms, clean_sms = clean_list("SMS Names2")
original_zoho, clean_zoho = clean_list("Zoho")

choices_clean = {
    "Radhika": clean_radhika,
    "Automation": clean_aut,
    "SMS": clean_sms,
    "Zoho": clean_zoho,
}

choices_original = {
    "Radhika": original_radhika,
    "Automation": original_aut,
    "SMS": original_sms,
    "Zoho": original_zoho,
}

thresholds = {"Radhika": 80, "Automation": 80, "SMS": 80, "Zoho": 80}

# ============================================================
# 6. Run Matching
# ============================================================
results = []

for name in df["Main Names"]:
    row = {"Main Name": name}

    for col in choices_clean:
        match, score, note = find_best_match(
            name,
            choices_clean[col],
            choices_original[col],
            thresholds[col]
        )
        row[f"{col} Match"] = match
        row[f"{col} Score"] = score
        row[f"{col} Note"] = note

    results.append(row)

results_df = pd.DataFrame(results)
output_path = r"D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\python result\matched_results_easy.xlsx"
results_df.to_excel(output_path, index=False)

print("✅ Matching complete.")
print(f"Saved to: {output_path}")


✅ Matching complete.
Saved to: D:\OneDrive - TANLA PLATFORMS LIMITED\Desktop\python result\matched_results_easy.xlsx
